In [3]:
%pip install pandas
%pip install numpy
%pip install pylsl
%pip install matplotlib
%pip install scipy
%pip install torch

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
%matplotlib inline
from FingerAngles import FingerAngles
from matplotlib import pyplot as plt
from pylsl import StreamInlet, resolve_stream
import time 
import pandas as pd 

# Ignore UserWarning messages
# These are especially common with mediapipe
import warnings
# Ignore all UserWarning messages
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
finger_tracker = FingerAngles()
finger_tracker.start()

# To use the tracker, first call update() before each new frame, which returns the image

# you can then call the get_angle or get_percent methods.
# These methods return either the angle or percentage open given the finger:
# ("thumb", "index", "middle", "ring", "pinky")

# Don't forget to stop the tracker when you're done
# finger_tracker.stop()

I0000 00:00:1721944346.474845 4671765 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3 Max
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1721944346.479307 4671937 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1721944346.484272 4671935 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
2024-07-25 16:52:26.561 python[72295:4671765] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


In [3]:
# resolve an EMG stream on the lab network and notify the user
print("Looking for an EMG stream...")
streams = resolve_stream('type', 'EMG')
inlet = StreamInlet(streams[0])
print("EMG stream found!")

# create a dataframe to store the data
channel_list = [1, 2, 3, 4]
columns = ['timestamp'] + [f'ch{i}'for i in channel_list] + finger_tracker.fingers
data = pd.DataFrame(columns=columns)
start_time = time.time()
run_time = 10

# loop for 10 seconds
while time.time() - start_time < run_time:
    finger_tracker.update(draw_camera=False, terminal_out=False) # update finger angles
    if not finger_tracker.found_hand: # if hand is not found, skip this iteration
        print("Hand not found")
        continue
    
    sample, timestamp = inlet.pull_sample() # get EMG data sample and its timestamp
    angles_list = finger_tracker.get_angle_list()
    percent_list = finger_tracker.get_percent_list()
    
    
    data.loc[len(data)] = [timestamp] + [sample[ch-1] for ch in channel_list] + percent_list # append data to dataframe
    if len(data) % 50 == 0:
        print(data.tail(50)) # print the last row of the dataframe
    

Looking for an EMG stream...
EMG stream found!


2024-07-25 16:52:28.786 (   2.386s) [          474915]      netinterfaces.cpp:91    INFO| netif 'lo0' (status: 1, multicast: 32768, broadcast: 0)
2024-07-25 16:52:28.786 (   2.386s) [          474915]      netinterfaces.cpp:91    INFO| netif 'lo0' (status: 1, multicast: 32768, broadcast: 0)
2024-07-25 16:52:28.786 (   2.386s) [          474915]      netinterfaces.cpp:102   INFO| 	IPv4 addr: 7f000001
2024-07-25 16:52:28.786 (   2.386s) [          474915]      netinterfaces.cpp:91    INFO| netif 'lo0' (status: 1, multicast: 32768, broadcast: 0)
2024-07-25 16:52:28.786 (   2.386s) [          474915]      netinterfaces.cpp:105   INFO| 	IPv6 addr: ::1
2024-07-25 16:52:28.787 (   2.386s) [          474915]      netinterfaces.cpp:91    INFO| netif 'lo0' (status: 1, multicast: 32768, broadcast: 0)
2024-07-25 16:52:28.787 (   2.386s) [          474915]      netinterfaces.cpp:105   INFO| 	IPv6 addr: fe80::1%lo0
2024-07-25 16:52:28.787 (   2.386s) [          474915]      netinterfaces.cpp:91    I

        timestamp       ch1       ch2       ch3       ch4  thumb  index  \
0   334919.122678  0.021786  0.026731  0.020035  0.061803    0.0    0.0   
1   334919.126678  0.035658  0.046172  0.023731  0.105225    0.0    0.0   
2   334919.130678  0.020224  0.025041  0.016605  0.049461    0.0    0.0   
3   334919.134678  0.014925  0.017914  0.014359  0.029251    0.0    0.0   
4   334919.138678  0.012343  0.014148  0.016916  0.068882    0.0    0.0   
5   334919.142678  0.012378  0.014242  0.016917  0.068729    0.0    0.0   
6   334919.146678  0.012435  0.014320  0.017682  0.070064    0.0    0.0   
7   334919.150678  0.012514  0.014347  0.016834  0.069375    0.0    0.0   
8   334919.153244  0.023381  0.028409  0.019034  0.058702    0.0    0.0   
9   334919.157244  0.036874  0.048181  0.023899  0.099757    0.0    0.0   
10  334919.161244  0.021885  0.027294  0.017199  0.051034    0.0    0.0   
11  334919.165244  0.016292  0.019834  0.014036  0.030883    0.0    0.0   
12  334919.169244  0.0125

In [4]:
data

,timestamp,ch1,ch2,ch3,ch4,thumb,index,middle,ring,pinky
0,334919.122678,0.021786,0.026731,0.020035,0.061803,0.0,0.000000,0.000000,0.000000,8.901258
1,334919.126678,0.035658,0.046172,0.023731,0.105225,0.0,0.000000,0.000000,5.178744,19.346036
2,334919.130678,0.020224,0.025041,0.016605,0.049461,0.0,0.000000,0.000000,7.559664,23.229034
3,334919.134678,0.014925,0.017914,0.014359,0.029251,0.0,0.000000,0.057405,11.669263,27.880485
4,334919.138678,0.012343,0.014148,0.016916,0.068882,0.0,0.000000,1.431455,13.943940,29.525108
...,...,...,...,...,...,...,...,...,...,...
236,334920.003722,0.014140,0.017810,0.026630,0.039577,0.0,18.437313,14.614981,5.908797,5.952344
237,334920.007722,0.014246,0.017890,0.025840,0.040607,0.0,4.421232,6.558057,1.579767,3.575621
238,334920.011722,0.014295,0.018016,0.026998,0.040491,0.0,3.314271,4.885683,0.962597,1.515106
239,334920.015722,0.014244,0.018029,0.027493,0.040321,0.0,1.847353,3.226852,0.610317,0.128026
